# Djinni Step 3 Cleanup

Notebook này làm sạch logic review của bước 3 từ file Round 2, rồi xuất ra file review đã sửa.

In [8]:
import pandas as pd
import numpy as np

INPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/07_djinni_step3_review.xlsx"
OUTPUT_FILE = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/07_djinni_step3_review_fixed_v2.xlsx"

# đọc sheet chính
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1").copy()

# đảm bảo có row_id
if "row_id" not in df.columns:
    df["row_id"] = range(2, len(df) + 2)

# chuẩn hoá text
for col in [
    "huong_xu_ly",
    "ten_goc",
    "ten_sach",
    "ten_ngoai_thi_truong",
    "cac_ten_gan_giong",
    "review_issue",
]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str).str.strip()

# tạo lại review_issue từ đầu cho sạch
df["review_issue"] = ""

mask_missing_market_name = (
    df["huong_xu_ly"].eq("doi_ten")
    & df["ten_ngoai_thi_truong"].eq("")
)
df.loc[mask_missing_market_name, "review_issue"] = "thieu_ten_ngoai_thi_truong"

mask_missing_alias = (
    df["huong_xu_ly"].eq("them_ten_gan_giong")
    & df["cac_ten_gan_giong"].eq("")
)
df.loc[mask_missing_alias, "review_issue"] = np.where(
    df.loc[mask_missing_alias, "review_issue"].eq(""),
    "thieu_cac_ten_gan_giong",
    df.loc[mask_missing_alias, "review_issue"] + "; thieu_cac_ten_gan_giong"
)

market_name_counts = (
    df.loc[df["ten_ngoai_thi_truong"].ne(""), "ten_ngoai_thi_truong"]
    .value_counts()
)
duplicate_market_names = set(market_name_counts[market_name_counts > 1].index)

mask_duplicate_market_name = (
    df["ten_ngoai_thi_truong"].isin(duplicate_market_names)
    & df["ten_ngoai_thi_truong"].ne("")
)

df.loc[mask_duplicate_market_name, "review_issue"] = np.where(
    df.loc[mask_duplicate_market_name, "review_issue"].eq(""),
    "trung_ten_ngoai_thi_truong",
    df.loc[mask_duplicate_market_name, "review_issue"] + "; trung_ten_ngoai_thi_truong"
)

# cờ xem thủ công
df["can_xem_thu_cong"] = np.where(
    df["huong_xu_ly"].isin(["doi_ten", "them_ten_gan_giong"]),
    1,
    0
)

# ưu tiên review
df["review_priority"] = np.where(
    df["review_issue"].ne(""),
    "issue",
    np.where(df["can_xem_thu_cong"].eq(1), "can_xem", "")
)

# tạo lại review sheet
review_df = df[
    (df["can_xem_thu_cong"].eq(1)) | (df["review_issue"].ne(""))
].copy()

priority_order = {"issue": 0, "can_xem": 1}
review_df["priority_rank"] = review_df["review_priority"].map(priority_order).fillna(99)

review_cols_first = [
    "row_id",
    "huong_xu_ly",
    "review_priority",
    "review_issue",
    "ten_goc",
    "ten_sach",
    "ten_ngoai_thi_truong",
    "cac_ten_gan_giong",
    "ten_thi_truong",
    "ten_gan_giong",
]

other_cols = [c for c in review_df.columns if c not in review_cols_first + ["priority_rank"]]

review_df = (
    review_df[review_cols_first + other_cols + ["priority_rank"]]
    .sort_values(by=["priority_rank", "row_id"], ascending=[True, True])
    .drop(columns=["priority_rank"])
    .reset_index(drop=True)
)

# summary sạch
summary = {
    "tong_so_dong": int(len(df)),
    "giu_nguyen": int((df["huong_xu_ly"] == "giu_nguyen").sum()),
    "doi_ten": int((df["huong_xu_ly"] == "doi_ten").sum()),
    "them_ten_gan_giong": int((df["huong_xu_ly"] == "them_ten_gan_giong").sum()),
    "dong_can_xem_thu_cong": int(df["can_xem_thu_cong"].sum()),
    "dong_co_issue": int((df["review_issue"] != "").sum()),
    "issue_thieu_ten_ngoai_thi_truong": int(mask_missing_market_name.sum()),
    "issue_thieu_cac_ten_gan_giong": int(mask_missing_alias.sum()),
    "issue_trung_ten_ngoai_thi_truong": int(mask_duplicate_market_name.sum()),
    "so_ten_ngoai_thi_truong_bi_trung_unique": int(len(duplicate_market_names)),
}
summary_df = pd.DataFrame(list(summary.items()), columns=["chi_so", "gia_tri"])

# ghi file mới
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")
    review_df.to_excel(writer, index=False, sheet_name="review_buoc_3")
    summary_df.to_excel(writer, index=False, sheet_name="summary_buoc_3")

print(f"Đã tạo file: {OUTPUT_FILE}")
print(summary)

Đã tạo file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/Djinni/notebook_clean/07_djinni_step3_review_fixed_v2.xlsx
{'tong_so_dong': 1171, 'giu_nguyen': 1116, 'doi_ten': 46, 'them_ten_gan_giong': 9, 'dong_can_xem_thu_cong': 55, 'dong_co_issue': 16, 'issue_thieu_ten_ngoai_thi_truong': 0, 'issue_thieu_cac_ten_gan_giong': 0, 'issue_trung_ten_ngoai_thi_truong': 16, 'so_ten_ngoai_thi_truong_bi_trung_unique': 8}
